## Enhancing Public Safety: A Comprehensive Analysis of Geospatial Shifts in Los Angeles and Prediction of Crime Volatility for LAPD Deployment

### Team
Our team, Treeo, consists of three members:
- Ashley Rauch will act as the POC for the group - ashrauch4
- Madeyln Forster - mgforste
- Brealin Redecker - brealinredecker

### Overview
Our crime reporting dataset reflects all incidents of crime in the City of Los Angeles dating back to 2020, collected by The Los Angeles Police Department (LAPD). Ensuring the safety of the public is a fundamental responsibility of the government. Crime, of all levels, can lead to loss of life, physical injury, and significant economic and social impacts on individuals and communities. While a certain level of crime may be considered inevitable, we aim to uncover structural shifts in crime patterns and build a tool that predicts crime volatility and potential "hotspots". The tool will enable the LAPD to develop evidence-based strategies for deployment of its forces as there is a finite number of patrol units available. 

Most crime prediction models explore static relationships, such as linking demographic data to crime rates. These approaches of predictive policing tend to reinforce overpolicing in areas that already get heavy surveillance. Hence, historical "hotspots" are not enough and dismiss crucial aspects of racial biases, current deployment, and accuracy in crime reporting. Long term, instead of just predicting emerging high-volatility crime areas, we are framing it as a deployment question: where should the LAPD actually field its forces? Shifting our focus slightly was important as currently, areas with less police presence result in fewer crime reportings and vice versa, so a model trained only on reported data will keep sending resources to the same places and neglecting everywhere else. We hope to bring in additional data on historical deployment, demographic information on the area, and/or NYC crime data to perform a comparative analysis. Furthermore, we plan to build in an adjustment that estimated how police presence affects the likelihood that a crime actually gets reported and checks against over-concentration and setting minimum coverage thresholds so no area gets completely ignored.

As briefly mentioned, the primary stakeholder for this project is the LAPD. The LAPD are responsible for public safety, and crime incidents are a major public safety concern as they aim to reduce the number of crimes in the areas they police. Success in this project will make two main differences:
1. Ensure resources don't just get locked in historical patterns.
2. Distinguish table high-crime zones from emergencying high-volatility areas so police forces can be adequately deployed.

Overall, our approach and specific needs have not changed. Our team, though, has done some deeper thinking on how we can incorporate other data to improve our model from just predicting hotspots.

### Data
Link to Data: "https://catalog.data.gov/dataset/crime-data-from-2020-to-present"

Our project will utilize data from 'data.gov' and is a crime data set that gives crime information from 2020-Present in Los Angeles. The data has 28 columns, a combination of string, integer, and float features, with over a million rows of data. As the data is government data, it must be accurate, ensuring it is reliable for model use. Also, according to the Freedom of Information Act, they are legally obligated to disclose information — including crime data and records of misconduct — under public records laws. The data includes meta deta, such as that it was published by the Los Angeles Police Department (LAPD), it is publically available data, and it was last updated on January 2, 2026. Some additional aspects we have noted will impact our model, though, is the data is noisy and the specific times when and where crimes get reported versus logged is not always reliable. Many times they will be inputted into the system at the location of the police station in batches of times - this unknown, while can't be changed, needs to be considered when analyzing the model.

### Exploratory Data Analysis
Before completing any exploratory data analysis, we filtered the dataframe to only include crime codes corresponding to assaults, as that is a prominent crime in LA and reduces safety in the city. As noted above the time corresponding to when the crime was reported and occurred was inputted as 12AM. For the date columns, we included only the date and converted the time column to a standard time format. Finally, we removed invalid coordinates to ensure our spatial framework works properly. Our group used this cleaned dataset to explore distributions and patterns.

In [4]:
%%capture
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder

# Load in the data
data = pd.read_csv("LA_Assault_Data_2020_Present.csv")
data.info() # quick examination of the data

In [5]:
#We noticed our data had a lot of null values, so we investigated further by looking at the percentage of nulls in each column.
    #As you can see, columns "Crm Cd 2", "Crm Cd 3" and "Crm Cd 4" all had a null percentantage above 90%. 
    #We realized this data is being captured in the "Crm Cd" and "Crm Cd 1" column because those have a null percentage of 0%. 
    #Because of this, we are okay with deleting this columns to ensure maximum efficiency when loading our data. 
#"Cross Street" also had a high null percentage of 84.65%. 
    #We realized we have other columns that capture the location, so like the Crm Cds, we are okay with deleting this column as well.
#"Weapon Used Cd" and "Weapon Desc" also had a high null percentage of 67.44% each. 
    #Because there's no other column that has weapon used, we are going to keep those columns in our dataset, but we aren't going to heavily rely on it.
    #Our project also focuses on how to deploy forces and reporting rates, so having a high null percentage for the weapons column is okay. 
data.isnull().sum()
print((data.isnull().sum() / len(data) * 100).round(2))

DR_NO              0.00
Date Rptd          0.00
DATE OCC           0.00
TIME OCC           0.00
AREA               0.00
AREA NAME          0.00
Rpt Dist No        0.00
Part 1-2           0.00
Crm Cd             0.00
Crm Cd Desc        0.00
Mocodes           15.09
Vict Age           0.00
Vict Sex          14.39
Vict Descent      14.39
Premis Cd          0.00
Premis Desc        0.06
Weapon Used Cd    67.44
Weapon Desc       67.44
Status             0.00
Status Desc        0.00
Crm Cd 1           0.00
Crm Cd 2          93.12
Crm Cd 3          99.77
Crm Cd 4          99.99
LOCATION           0.00
Cross Street      84.65
LAT                0.00
LON                0.00
dtype: float64


In [6]:
#Dropping the columns discussed in the previous cell. 
threshold = 80
columns_to_drop = (data.isnull().sum() / len(data) * 100)[lambda x: x > threshold].index
data_cleaned = data.drop(columns=columns_to_drop)
print(f"Dropped columns: {list(columns_to_drop)}")
print(f"Remaining columns count: {data_cleaned.shape[1]}")

Dropped columns: ['Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'Cross Street']
Remaining columns count: 24


In [7]:
#What's remaining
print("Columns with remaining nulls:")
print(data_cleaned.isnull().sum()[data_cleaned.isnull().sum() > 0])

print("\n" + "#"*60 + "\n")
print("Percentage of nulls:")
print((data_cleaned.isnull().sum() / len(data_cleaned) * 100)[lambda x: x > 0].round(2))

Columns with remaining nulls:
Mocodes           151619
Vict Sex          144644
Vict Descent      144656
Premis Cd             16
Premis Desc          588
Weapon Used Cd    677744
Weapon Desc       677744
Status                 1
Crm Cd 1              11
dtype: int64

############################################################

Percentage of nulls:
Mocodes           15.09
Vict Sex          14.39
Vict Descent      14.39
Premis Cd          0.00
Premis Desc        0.06
Weapon Used Cd    67.44
Weapon Desc       67.44
Status             0.00
Crm Cd 1           0.00
dtype: float64


In [8]:
#Converting our variables into dummy variables
encoder = OneHotEncoder()
variables_one_hot = encoder.fit_transform(data_cleaned[['Vict Sex']])
df_one_hot = pd.DataFrame(variables_one_hot, columns=encoder.get_feature_names_out(['Vict Sex']))

ValueError: Shape of passed values is (1004991, 1), indices imply (1004991, 6)